# DetoxifyAI RAG Pipeline
## Milestone 2 - D2: Retrieval-Augmented Generation

This notebook implements a RAG pipeline for toxic message rephrasing using:
- **Knowledge Base**: 200+ toxic → professional examples + style guides
- **Retrieval**: FAISS vector search with sentence-transformers
- **Generation**: LLM with few-shot prompting (k=5)
- **Framework**: LangChain for modularity
- **Storage**: Azure Blob Storage for artifacts

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q azure-storage-blob
!pip install -q transformers accelerate bitsandbytes
!pip install -q python-dotenv

In [ ]:
# Import libraries
import json
import pandas as pd
from typing import List, Dict
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# LangChain
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document

In [ ]:
# Hugging Face
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [ ]:
# Azure Storage
from azure.storage.blob import BlobServiceClient
import pickle

## 2. Configuration

In [ ]:
# Configuration
CONFIG = {
    # Model Configuration
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",  # Same as BibleRAG
    "llm_model": "mistralai/Mistral-7B-Instruct-v0.1",  # For local testing
    # RAG Configuration
    "top_k": 5,  # Number of examples to retrieve for few-shot
    "max_length": 512,  # Max tokens for generation
    "temperature": 0.7,
    # File Paths (upload these to Colab)
    "knowledge_base_path": "rag_knowledge_base.jsonl",
    "style_guide_path": "style_guide_professional_communication.txt",
    "templates_path": "workplace_templates.txt",
    "categories_path": "category_rephrasing_examples.txt",
    # Azure Blob Storage
    "azure_connection_string": "YOUR_AZURE_CONNECTION_STRING",  # TODO: Add your connection string
    "azure_container_name": "detoxifyai-artifacts",
    "faiss_index_blob_name": "faiss_index.pkl",
    "knowledge_base_blob_name": "knowledge_base.pkl",
}

print("✅ Configuration loaded")
print(f"Embedding Model: {CONFIG['embedding_model']}")
print(f"LLM Model: {CONFIG['llm_model']}")
print(f"Top-K Examples: {CONFIG['top_k']}")

## 3. Data Loading & Preprocessing
Following BibleRAG structure: Load and prepare knowledge base

In [ ]:
# Load knowledge base (200 toxic -> professional examples)
def load_knowledge_base(file_path: str) -> List[Dict]:
    """
    Load toxic -> professional rephrasing examples from JSONL
    Similar to Bible verse loading in BibleRAG
    """
    examples = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            examples.append(json.loads(line))
    return examples


# Load examples
knowledge_base = load_knowledge_base(CONFIG["knowledge_base_path"])

print(f"✅ Loaded {len(knowledge_base)} rephrasing examples")
print("\nExample entry:")
print(json.dumps(knowledge_base[0], indent=2))

In [ ]:
# Load style guides and templates
def load_text_file(file_path: str) -> str:
    """Load text-based knowledge documents"""
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


style_guide = load_text_file(CONFIG["style_guide_path"])
templates = load_text_file(CONFIG["templates_path"])
categories = load_text_file(CONFIG["categories_path"])

print("✅ Loaded supplementary documents")
print(f"Style Guide: {len(style_guide)} characters")
print(f"Templates: {len(templates)} characters")
print(f"Categories: {len(categories)} characters")

In [ ]:
# Create DataFrame for analysis (similar to BibleRAG)
df = pd.DataFrame(knowledge_base)

print("\n📊 Knowledge Base Statistics:")
print(f"Total Examples: {len(df)}")
print("\nCategories Distribution:")
print(df["category"].value_counts().head(10))
print("\nContext Distribution:")
print(df["context"].value_counts().head(10))

In [ ]:
# Prepare documents for embedding (combining toxic + professional for better retrieval)
def prepare_documents(examples: List[Dict]) -> List[Document]:
    """
    Convert examples to LangChain Document format
    Similar to verse preparation in BibleRAG
    """
    documents = []

    for ex in examples:
        # Create searchable text: combine toxic input with professional output
        # This allows retrieval based on similarity to toxic input
        content = f"Toxic: {ex['toxic']}\nProfessional: {ex['professional']}"

        # Metadata for filtering and context
        metadata = {
            "id": ex["id"],
            "category": ex["category"],
            "context": ex["context"],
            "toxic": ex["toxic"],
            "professional": ex["professional"],
        }

        documents.append(Document(page_content=content, metadata=metadata))

    return documents


documents = prepare_documents(knowledge_base)

print(f"✅ Prepared {len(documents)} documents for embedding")
print("\nSample document:")
print(f"Content: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

## 4. Embedding Generation
Using sentence-transformers (same as BibleRAG)

In [ ]:
# Initialize embedding model
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # Important for cosine similarity
)

print(f"✅ Embedding model loaded: {CONFIG['embedding_model']}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Test embedding
test_text = "You're an idiot"
test_embedding = embedding_model.embed_query(test_text)

print("✅ Embedding test successful")
print(f"Text: '{test_text}'")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"Embedding sample: {test_embedding[:5]}")

## 5. FAISS Index Creation
Following BibleRAG structure with LangChain integration

In [ ]:
# Create FAISS vector store (LangChain wrapper)
print("Creating FAISS index...")
print("This may take a few minutes for 200 examples...")

vectorstore = FAISS.from_documents(documents=documents, embedding=embedding_model)

print(f"✅ FAISS index created with {len(documents)} documents")

In [ ]:
# Test retrieval
query = "You're completely useless at your job"
retrieved_docs = vectorstore.similarity_search(query, k=5)

print("\n🔍 Retrieval Test")
print(f"Query: '{query}'")
print("\nTop-5 Retrieved Examples:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n{i}. Category: {doc.metadata['category']}")
    print(f"   Toxic: {doc.metadata['toxic']}")
    print(f"   Professional: {doc.metadata['professional']}")

## 6. Save FAISS Index Locally
Before uploading to Azure Blob Storage

In [ ]:
# Save FAISS index locally
local_faiss_path = "faiss_index_local"
vectorstore.save_local(local_faiss_path)

print(f"✅ FAISS index saved locally to: {local_faiss_path}")

# Also save knowledge base for reconstruction
with open("knowledge_base.pkl", "wb") as f:
    pickle.dump(knowledge_base, f)

print("✅ Knowledge base saved to: knowledge_base.pkl")

## 7. Azure Blob Storage Integration
Upload FAISS index to Azure (similar to your M1 setup)

In [ ]:
# Azure Blob Storage Helper Functions
def upload_to_azure_blob(local_file_path: str, blob_name: str, connection_string: str, container_name: str):
    """
    Upload file to Azure Blob Storage
    """
    try:
        blob_service_client = BlobServiceClient.from_connection_string(connection_string)
        blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)

        with open(local_file_path, "rb") as data:
            blob_client.upload_blob(data, overwrite=True)

        print(f"✅ Uploaded {local_file_path} to Azure Blob: {blob_name}")
        return True
    except Exception as e:
        print(f"❌ Error uploading to Azure: {e}")
        return False


def download_from_azure_blob(blob_name: str, local_file_path: str, connection_string: str, container_name: str):
    """
    Download file from Azure Blob Storage
    """
    try:
        blob_service_client = BlobServiceClient.from_connection_string(connection_string)
        blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)

        with open(local_file_path, "wb") as data:
            data.write(blob_client.download_blob().readall())

        print(f"✅ Downloaded {blob_name} from Azure Blob to: {local_file_path}")
        return True
    except Exception as e:
        print(f"❌ Error downloading from Azure: {e}")
        return False

In [ ]:
# Upload FAISS index to Azure
# NOTE: Update CONFIG['azure_connection_string'] with your actual connection string

if CONFIG["azure_connection_string"] != "YOUR_AZURE_CONNECTION_STRING":
    # Create container if it doesn't exist
    try:
        blob_service_client = BlobServiceClient.from_connection_string(CONFIG["azure_connection_string"])
        container_client = blob_service_client.get_container_client(CONFIG["azure_container_name"])

        if not container_client.exists():
            container_client.create_container()
            print(f"✅ Created Azure container: {CONFIG['azure_container_name']}")
    except Exception as e:
        print(f"⚠️ Container creation: {e}")

    # Upload FAISS index files
    import shutil

    shutil.make_archive("faiss_index", "zip", local_faiss_path)

    upload_to_azure_blob(
        "faiss_index.zip", "faiss_index.zip", CONFIG["azure_connection_string"], CONFIG["azure_container_name"]
    )

    # Upload knowledge base
    upload_to_azure_blob(
        "knowledge_base.pkl",
        CONFIG["knowledge_base_blob_name"],
        CONFIG["azure_connection_string"],
        CONFIG["azure_container_name"],
    )
else:
    print("⚠️ Please update CONFIG['azure_connection_string'] with your actual connection string")
    print("Skipping Azure upload for now...")

## ✅ Checkpoint: Data & Retrieval Ready

**What we've accomplished:**
1. ✅ Loaded 200 toxic → professional examples
2. ✅ Created embeddings with sentence-transformers
3. ✅ Built FAISS index for fast retrieval
4. ✅ Tested retrieval (top-k similar examples)
5. ✅ Saved artifacts locally
6. ✅ Uploaded to Azure Blob Storage

**Next steps:** Few-shot prompt generation + LLM integration

---
# Part 2: Few-Shot Prompting + LLM Integration

## 8. Few-Shot Prompt Builder (k=5)
Build prompts using retrieved examples from D1 strategy

In [ ]:
# Few-Shot Prompt Template (from D1 experiments)
FEW_SHOT_TEMPLATE = """You are a professional communication expert. Your task is to rephrase toxic messages into polite, professional alternatives while preserving the original intent.

Here are some examples of toxic messages transformed into professional communication:

{examples}

Now, please rephrase the following toxic message into a professional alternative:

Toxic Message: {toxic_input}

Professional Rephrase:"""

print("✅ Few-shot prompt template loaded")

In [ ]:
def build_few_shot_prompt(toxic_input: str, retrieved_docs: List[Document], k: int = 5) -> str:
    """
    Build few-shot prompt with k retrieved examples

    Args:
        toxic_input: The toxic message to rephrase
        retrieved_docs: List of retrieved similar examples from FAISS
        k: Number of examples to include (default 5)

    Returns:
        Complete prompt with examples
    """
    # Format examples
    examples_text = ""
    for i, doc in enumerate(retrieved_docs[:k], 1):
        examples_text += f"""Example {i}:
Toxic: "{doc.metadata["toxic"]}"
Professional: "{doc.metadata["professional"]}"

"""

    # Build complete prompt
    prompt = FEW_SHOT_TEMPLATE.format(examples=examples_text.strip(), toxic_input=toxic_input)

    return prompt


print("✅ Few-shot prompt builder ready")

In [ ]:
# Test prompt building
test_toxic = "You're an idiot for making that mistake"
test_retrieved = vectorstore.similarity_search(test_toxic, k=5)
test_prompt = build_few_shot_prompt(test_toxic, test_retrieved, k=5)

print("🔍 Test Prompt Generated:")
print("=" * 80)
print(test_prompt)
print("=" * 80)

## 9. LLM Integration (Local Testing)
Load Mistral-7B with 4-bit quantization for Colab

In [ ]:
# Check GPU availability
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Load LLM with 4-bit quantization (fits in Colab T4 GPU)
print("Loading Mistral-7B-Instruct with 4-bit quantization...")
print("This may take 2-3 minutes...")

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model_name = CONFIG["llm_model"]
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)

print(f"✅ Model loaded: {model_name}")
print("Model size: ~4 GB (quantized from ~14 GB)")

In [ ]:
# Generation function
def generate_rephrase(prompt: str, max_length: int = 512, temperature: float = 0.7) -> str:
    """
    Generate professional rephrase using LLM

    Args:
        prompt: Complete few-shot prompt
        max_length: Maximum tokens to generate
        temperature: Sampling temperature (0.7 for creativity)

    Returns:
        Generated professional rephrase
    """
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the rephrased part (after "Professional Rephrase:")
    if "Professional Rephrase:" in generated_text:
        rephrased = generated_text.split("Professional Rephrase:")[-1].strip()
    else:
        rephrased = generated_text[len(prompt) :].strip()

    return rephrased


print("✅ Generation function ready")

In [ ]:
# Test generation
print("🧪 Testing LLM generation...\n")

test_output = generate_rephrase(test_prompt, max_length=150, temperature=0.7)

print("Original Toxic:", test_toxic)
print("\nGenerated Professional Rephrase:")
print(test_output)

## 10. End-to-End RAG Pipeline
Complete function combining retrieval + generation

In [ ]:
def rag_rephrase_pipeline(toxic_input: str, k: int = 5, verbose: bool = True) -> Dict:
    """
    End-to-end RAG pipeline for toxic message rephrasing

    Args:
        toxic_input: Toxic message to rephrase
        k: Number of examples to retrieve
        verbose: Print intermediate steps

    Returns:
        Dictionary with original, rephrased, and metadata
    """
    if verbose:
        print(f"\n{'=' * 80}")
        print("RAG PIPELINE STARTED")
        print(f"{'=' * 80}\n")
        print(f"Input: {toxic_input}\n")

    # Step 1: Retrieve similar examples
    if verbose:
        print("Step 1: Retrieving similar examples from FAISS...")

    retrieved_docs = vectorstore.similarity_search(toxic_input, k=k)

    if verbose:
        print(f"✅ Retrieved {len(retrieved_docs)} examples")
        print("\nTop 3 similar examples:")
        for i, doc in enumerate(retrieved_docs[:3], 1):
            print(f"  {i}. {doc.metadata['toxic'][:60]}...")

    # Step 2: Build few-shot prompt
    if verbose:
        print("\nStep 2: Building few-shot prompt...")

    prompt = build_few_shot_prompt(toxic_input, retrieved_docs, k=k)

    if verbose:
        print(f"✅ Prompt built with {k} examples")
        print(f"Prompt length: {len(prompt)} characters")

    # Step 3: Generate rephrase
    if verbose:
        print("\nStep 3: Generating professional rephrase...")

    rephrased = generate_rephrase(prompt, max_length=CONFIG["max_length"], temperature=CONFIG["temperature"])

    if verbose:
        print("✅ Generation complete")

    # Prepare result
    result = {
        "toxic_input": toxic_input,
        "professional_rephrase": rephrased,
        "retrieved_examples": [
            {
                "toxic": doc.metadata["toxic"],
                "professional": doc.metadata["professional"],
                "category": doc.metadata["category"],
            }
            for doc in retrieved_docs[:k]
        ],
        "num_examples_used": k,
    }

    if verbose:
        print(f"\n{'=' * 80}")
        print("RESULT")
        print(f"{'=' * 80}")
        print("\n📝 Original (Toxic):")
        print(f"   {toxic_input}")
        print("\n✨ Rephrased (Professional):")
        print(f"   {rephrased}")
        print(f"\n{'=' * 80}\n")

    return result


print("✅ End-to-end RAG pipeline ready")

## 11. Pipeline Testing
Test with various toxic message types

In [ ]:
# Test cases from different categories
test_cases = [
    "You're an idiot for making that mistake",
    "This is complete garbage, redo it",
    "I don't care about your excuses, just get it done",
    "What a stupid idea, that will never work",
    "You're completely useless at your job",
]

print("\n🧪 TESTING RAG PIPELINE ON MULTIPLE EXAMPLES\n")
print("=" * 80)

In [ ]:
# Run pipeline on each test case
results = []

for i, test_case in enumerate(test_cases, 1):
    print(f"\n\n{'#' * 80}")
    print(f"TEST CASE {i}/{len(test_cases)}")
    print(f"{'#' * 80}")

    result = rag_rephrase_pipeline(test_case, k=5, verbose=True)
    results.append(result)

    # Small delay to avoid overwhelming GPU
    import time

    time.sleep(1)

In [ ]:
# Summary of all results
print("\n\n" + "=" * 80)
print("SUMMARY OF ALL RESULTS")
print("=" * 80 + "\n")

for i, result in enumerate(results, 1):
    print(f"{i}. TOXIC: {result['toxic_input']}")
    print(f"   PROFESSIONAL: {result['professional_rephrase']}")
    print()

## 12. Save Results for Evaluation

In [ ]:
# Save results to JSON for evaluation
import json
from datetime import datetime

output_data = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "embedding_model": CONFIG["embedding_model"],
        "llm_model": CONFIG["llm_model"],
        "top_k": CONFIG["top_k"],
        "temperature": CONFIG["temperature"],
    },
    "results": results,
}

with open("rag_pipeline_results.json", "w") as f:
    json.dump(output_data, f, indent=2)

print("✅ Results saved to: rag_pipeline_results.json")

## 13. LangChain Integration (Production-Ready)
Wrap pipeline in LangChain for SageMaker deployment

In [ ]:
from langchain.llms.base import LLM
from typing import Optional, List, Any


class DetoxifyRAG(LLM):
    """
    Custom LangChain LLM wrapper for DetoxifyAI RAG pipeline
    This will be used for SageMaker deployment
    """

    vectorstore: Any
    model: Any
    tokenizer: Any
    k: int = 5
    temperature: float = 0.7
    max_length: int = 512

    @property
    def _llm_type(self) -> str:
        return "detoxify_rag"

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        """
        Main inference method
        """
        # This method expects the toxic input directly
        result = rag_rephrase_pipeline(toxic_input=prompt, k=self.k, verbose=False)
        return result["professional_rephrase"]

    @property
    def _identifying_params(self) -> dict:
        """Return identifying parameters."""
        return {"k": self.k, "temperature": self.temperature, "max_length": self.max_length}


# Create LangChain wrapper
detoxify_rag_llm = DetoxifyRAG(
    vectorstore=vectorstore,
    model=model,
    tokenizer=tokenizer,
    k=CONFIG["top_k"],
    temperature=CONFIG["temperature"],
    max_length=CONFIG["max_length"],
)

print("✅ LangChain wrapper created")

In [ ]:
# Test LangChain wrapper
test_input = "You're terrible at this job"
output = detoxify_rag_llm(test_input)

print("\n🧪 Testing LangChain Wrapper:")
print(f"Input: {test_input}")
print(f"Output: {output}")

## 14. Prepare for SageMaker Deployment
Save model artifacts for cloud deployment

In [ ]:
# Save deployment config
deployment_config = {
    "model_name": CONFIG["llm_model"],
    "embedding_model": CONFIG["embedding_model"],
    "top_k": CONFIG["top_k"],
    "temperature": CONFIG["temperature"],
    "max_length": CONFIG["max_length"],
    "quantization": "4-bit",
    "azure_blob_container": CONFIG["azure_container_name"],
    "faiss_index_blob": "faiss_index.zip",
    "knowledge_base_blob": CONFIG["knowledge_base_blob_name"],
}

with open("deployment_config.json", "w") as f:
    json.dump(deployment_config, f, indent=2)

print("✅ Deployment config saved: deployment_config.json")
print("\nThis config will be used for SageMaker deployment")

## ✅ RAG Pipeline Complete!

### What We've Built:
1. ✅ Data loading & preprocessing (200 examples)
2. ✅ FAISS vector index for retrieval
3. ✅ Few-shot prompt builder (k=5)
4. ✅ LLM integration (Mistral-7B, 4-bit quantized)
5. ✅ End-to-end RAG pipeline
6. ✅ LangChain wrapper for production
7. ✅ Azure Blob Storage integration
8. ✅ Testing & validation

### Artifacts Ready for Deployment:
- `faiss_index.zip` (in Azure Blob)
- `knowledge_base.pkl` (in Azure Blob)
- `deployment_config.json`
- `rag_pipeline_results.json`

### Next Steps:
1. **AWS SageMaker Setup** - Deploy LLM endpoint
2. **FastAPI Integration** - Connect to main app
3. **Evaluation** - Run on eval dataset
4. **Documentation** - Architecture diagrams

---

**Ready for AWS SageMaker deployment! 🚀**